# 🎓 SpeechAI: Training 6-Parameter Speech Fluency Scorer on Real Clinical Datasets
This notebook trains a **Wav2Vec2 Multi-Task Speech Fluency Scorer** to replace direct Gemini API calls.

### Datasets Analyzed & Unified:
- **UCLASS (`archive 8`)**: 4,712 clips of stuttered & fluent clinical speech
- **LibriStutter (`archive 9`)**: 4,736 full audio recordings with time-aligned stutter annotations
- **SEP-28k (`archive 10`)**: 28,177 audio clips with multi-annotator disfluency labels
- **Total Volume**: **37,625 high-quality annotated speech samples**

### 6 Parameters Predicted:
1. **`wpm`**: Speaking rate (Words Per Minute)
2. **`filler_count`**: Number of vocal fillers / interjections (*um, uh, like*)
3. **`stammer_count`**: Syllable repetitions, prolongations, and blocks
4. **`pause_count`**: Extended silent gaps & natural pauses
5. **`clarity`**: Acoustic articulation clarity (0.0 to 10.0 scale)
6. **`wer`**: Word Error Rate / Disfluency deviation (0.0 to 1.0 scale)

## 1. Environment Setup & GPU Check

In [ ]:
!nvidia-smi
!pip install -q torch torchaudio transformers scikit-learn pandas numpy scipy matplotlib soundfile librosa tqdm

## 2. Prepare & Unify Dataset
If you uploaded the archives or zip files, run the preprocessor to generate the unified `metadata.csv`.

In [ ]:
import os

# Unzip if uploaded as zip files
for z in ['dataset.zip', 'archive_8.zip', 'archive_9.zip', 'archive_10.zip']:
    if os.path.exists(z):
        print(f'Unzipping {z}...')
        !unzip -q -o {z}

# Run dataset unification engine
!python speech_scoring_model/prepare_unified_dataset.py \
    --dataset_dir dataset \
    --output_file metadata.csv \
    --max_samples 10000

import pandas as pd
df = pd.read_csv('dataset/metadata.csv')
print(f'\nReady to train on {len(df)} samples!')
df.head()

## 3. Train the Model
Trains the Wav2Vec2 backbone with Attentive Pooling and the 6-parameter multi-task head.

In [ ]:
!python speech_scoring_model/train.py \
    --data_dir dataset \
    --metadata_file metadata.csv \
    --output_dir checkpoints \
    --epochs 25 \
    --batch_size 16 \
    --device cuda \
    --fp16

## 4. Evaluate Test Metrics & Generate Research Plots

In [ ]:
!python speech_scoring_model/evaluate.py \
    --data_dir dataset \
    --metadata_file metadata.csv \
    --checkpoint checkpoints/best_model.pt \
    --scaler checkpoints/scaler.json \
    --output_dir evaluation_reports \
    --device cuda

from IPython.display import Image, display
if os.path.exists('evaluation_reports/prediction_correlation.png'):
    display(Image('evaluation_reports/prediction_correlation.png'))

## 5. Download Trained Model & Scaler for Your Project
Download `best_model.pt` and `scaler.json` and copy them into your `final_year_project/checkpoints/` folder.

In [ ]:
from google.colab import files
files.download('checkpoints/best_model.pt')
files.download('checkpoints/scaler.json')